<a href="https://colab.research.google.com/github/amcbhome/python-notebook/blob/main/Copy_of_ACCA_delivery_LP_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code using PuLP and to solve the ACCA delivery optimisation problem. Cleanly separates into four tasks by calling a function.

##1. Editing the variables

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np

# --- Default Data Definition (immutable) ---
# These values serve as the initial defaults and are not changed by the form.
DEFAULT_DISTANCE_MATRIX = np.array(
    [[
        22, 33, 40
    ],
     [
        27, 30, 22
    ],
     [
        36, 20, 25
    ]])
DEFAULT_STORE_CAPACITY = [
    2000, 3000, 2000
]
DEFAULT_DEPOT_CONSTRAINTS = [
    2500, 3100, 1250
]
# DEFAULT_DEMAND_CONSTRAINTS = [1000, 1500, 800] # Removed from form
DEFAULT_DELIVERY_COST_PER_UNIT = 5.00
DEFAULT_DEPOTS = [
    'Depot1', 'Depot2', 'Depot3'
]
DEFAULT_STORES = [
    'Store1', 'Store2', 'Store3'
]

# --- Current active variables (mutable, initially defaults) ---
# These variables will be updated by the form and used by the optimization function.
distance_matrix = DEFAULT_DISTANCE_MATRIX.copy()
store_capacity = list(DEFAULT_STORE_CAPACITY)
depot_constraints = list(DEFAULT_DEPOT_CONSTRAINTS)
demand_constraints = [
    0, 0, 0
] # Kept as placeholder, but not used in optimization after being commented out
delivery_cost_per_unit = DEFAULT_DELIVERY_COST_PER_UNIT
depots = list(DEFAULT_DEPOTS) # Not editable via form, modify directly if needed
stores = list(DEFAULT_STORES) # Not editable via form, modify directly if needed

# --- Widget Definitions ---

# Helper function to convert a numpy matrix to a user-friendly string for Textarea
def matrix_to_str(matrix):
    return "\n".join([",".join(map(str, row)) for row in matrix])

# Helper function to convert a list to a user-friendly string for Text widget
def list_to_str(lst):
    return ",".join(map(str, lst))

# Distance Matrix Textarea widget
distance_matrix_widget = widgets.Textarea(
    value=matrix_to_str(distance_matrix),
    description='Distance Matrix (comma-separated values, new line for rows):',
    layout=widgets.Layout(width='auto', height='100px'),
    continuous_update=False
)

# Store Capacity Text widget
store_capacity_widget = widgets.Text(
    value=list_to_str(store_capacity),
    description='Store Capacity (comma-separated):',
    layout=widgets.Layout(width='auto'),
    continuous_update=False
)

# Depot Constraints Text widget
depot_constraints_widget = widgets.Text(
    value=list_to_str(depot_constraints),
    description='Depot Constraints (comma-separated):',
    layout=widgets.Layout(width='auto'),
    continuous_update=False
)

# Removed Demand Constraints Text widget from form
# demand_constraints_widget = widgets.Text(
#     value=list_to_str(demand_constraints),
#     description='Demand Constraints (comma-separated):',
#     layout=widgets.Layout(width='auto'),
#     continuous_update=False
# )

# Delivery Cost per Unit FloatText widget
delivery_cost_widget = widgets.FloatText(
    value=delivery_cost_per_unit,
    description='Delivery Cost per Unit:',
    step=0.01,
    continuous_update=False
)

# Output widget to display messages (e.g., errors or success)
output_widget = widgets.Output()

# Update button widget
update_button = widgets.Button(description="Update Values")

# Restore Defaults button widget
restore_defaults_button = widgets.Button(description="Restore Defaults")

# --- Event Handler for the Update Button ---
def on_update_button_clicked(b):
    global distance_matrix, store_capacity, depot_constraints, demand_constraints, delivery_cost_per_unit

    with output_widget:
        output_widget.clear_output()
        print("Attempting to update values...")
        try:
            # Parse and validate Distance Matrix input
            dm_str = distance_matrix_widget.value
            rows = [list(map(float, row.strip().split(','))) for row in dm_str.split('\n') if row.strip()]
            new_distance_matrix = np.array(rows)
            # Validate dimensions against defaults (assuming fixed problem size)
            if new_distance_matrix.shape != DEFAULT_DISTANCE_MATRIX.shape:
                raise ValueError(f"Distance matrix must be {DEFAULT_DISTANCE_MATRIX.shape[0]}x{DEFAULT_DISTANCE_MATRIX.shape[1]}. Found {new_distance_matrix.shape}.")
            distance_matrix = new_distance_matrix

            # Parse and validate Store Capacity input
            sc_str = store_capacity_widget.value
            new_store_capacity = list(map(float, sc_str.split(',')))
            if len(new_store_capacity) != len(DEFAULT_STORE_CAPACITY):
                raise ValueError(f"Store capacity must have {len(DEFAULT_STORE_CAPACITY)} values. Found {len(new_store_capacity)}.")
            store_capacity = new_store_capacity

            # Parse and validate Depot Constraints input
            dc_str = depot_constraints_widget.value
            new_depot_constraints = list(map(float, dc_str.split(',')))
            if len(new_depot_constraints) != len(DEFAULT_DEPOT_CONSTRAINTS):
                raise ValueError(f"Depot constraints must have {len(DEFAULT_DEPOT_CONSTRAINTS)} values. Found {len(new_depot_constraints)}.")
            depot_constraints = new_depot_constraints

            # Removed parsing and validation for Demand Constraints
            # dd_str = demand_constraints_widget.value
            # new_demand_constraints = list(map(float, dd_str.split(',')))
            # if len(new_demand_constraints) != len(DEFAULT_DEMAND_CONSTRAINTS):
            #     raise ValueError(f"Demand constraints must have {len(DEFAULT_DEMAND_CONSTRAINTS)} values. Found {len(new_demand_constraints)}.")
            # demand_constraints = new_demand_constraints

            # Delivery Cost per Unit (no parsing needed, just assignment)
            delivery_cost_per_unit = delivery_cost_widget.value

            print("Values updated successfully! Please re-run the cell that calls the optimization function to see results with the new values.")

        except ValueError as e:
            print(f"Error parsing input: {e}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}. Please check your input format.")

# --- Event Handler for the Restore Defaults Button ---
def on_restore_defaults_button_clicked(b):
    global distance_matrix, store_capacity, depot_constraints, demand_constraints, delivery_cost_per_unit

    with output_widget:
        output_widget.clear_output()
        print("Restoring default values...")

        # Reset global variables to their defaults
        distance_matrix = DEFAULT_DISTANCE_MATRIX.copy()
        store_capacity = list(DEFAULT_STORE_CAPACITY)
        depot_constraints = list(DEFAULT_DEPOT_CONSTRAINTS)
        # demand_constraints = list(DEFAULT_DEMAND_CONSTRAINTS) # Removed from form
        delivery_cost_per_unit = DEFAULT_DELIVERY_COST_PER_UNIT

        # Update widget values to reflect defaults
        distance_matrix_widget.value = matrix_to_str(distance_matrix)
        store_capacity_widget.value = list_to_str(store_capacity)
        depot_constraints_widget.value = list_to_str(depot_constraints)
        # demand_constraints_widget.value = list_to_str(demand_constraints) # Removed from form
        delivery_cost_widget.value = delivery_cost_per_unit

        print("Default values restored to the form. Click 'Update Values' and re-run the optimization cell to apply them.")

# Attach the event handler to the buttons
update_button.on_click(on_update_button_clicked)
restore_defaults_button.on_click(on_restore_defaults_button_clicked)

# --- Display Widgets ---
display(HTML("<h3>Supply Chain Parameters</h3>"))
display(HTML("<p>Edit the values below and click 'Update Values' to apply changes. Then re-run the optimization cell.</p>"))

form_widgets = widgets.VBox([
    distance_matrix_widget,
    store_capacity_widget,
    depot_constraints_widget,
    # demand_constraints_widget, # Removed from form
    delivery_cost_widget,
    widgets.HBox([update_button, restore_defaults_button]), # Group buttons together
    output_widget
])

display(form_widgets)
display(HTML("<p><i>Note: Depot and Store names are currently fixed. To change them, please edit the 'depots' and 'stores' lists in the code directly.</i></p>"))

##2. Defining the function

In [ ]:
import numpy as np
import pandas as pd
import pulp

# --- Function Definition ---
def solve_supply_chain_optimization(distance_matrix, store_capacity, depot_constraints, demand_constraints, delivery_cost_per_unit, depots, stores):
    # Create a dictionary to map depot/store names to their index for matrix lookup
    depot_to_idx_func = {depot: i for i, depot in enumerate(depots)}
    store_to_idx_func = {store: i for i, store in enumerate(stores)}

    # Create the LP problem object
    prob = pulp.LpProblem("Supply_Chain_Optimization", pulp.LpMinimize)

    # Create decision variables: x[depot][store] represents the amount shipped
    x = pulp.LpVariable.dicts("Shipment", (depots, stores), 0, None, pulp.LpInteger)

    # Objective Function: Minimize total delivery cost
    prob += pulp.lpSum([x[d][s] * distance_matrix[depot_to_idx_func[d]][store_to_idx_func[s]] * delivery_cost_per_unit
                        for d in depots for s in stores]), "Total Delivery Cost"

    # 1. Depot Capacity Constraints (using equality as per previous user request)
    for i, d in enumerate(depots):
        prob += pulp.lpSum([x[d][s] for s in stores]) == depot_constraints[i], f"Depot_{d}_Capacity"

    # 2. Store Capacity Constraints
    for i, s in enumerate(stores):
        prob += pulp.lpSum([x[d][s] for d in depots]) <= store_capacity[i], f"Store_{s}_Capacity"

    # 3. Demand Constraints (Removed as per user request)
    # for i, s in enumerate(stores):
    #     prob += pulp.lpSum([x[d][s] for d in depots]) >= demand_constraints[i], f"Store_{s}_Demand"

    # Solve the problem
    prob.solve()

    optimal_shipments = {}
    total_cost = None
    status = pulp.LpStatus[prob.status]

    if status == 'Optimal':
        for d in depots:
            for s in stores:
                if x[d][s].varValue > 0:
                    optimal_shipments[f"Shipment_{d}_{s}"] = x[d][s].varValue
        total_cost = pulp.value(prob.objective)

    return status, optimal_shipments, total_cost

##3. Calling the function

In [ ]:
# Call the optimization function with the defined variables
status, optimal_shipments, total_cost = solve_supply_chain_optimization(
    distance_matrix,
    store_capacity,
    depot_constraints,
    demand_constraints,
    delivery_cost_per_unit,
    depots,
    stores
)

print(f"\nStatus: {status}")

if status == 'Optimal':
    print("\nOptimal Shipments:")
    for shipment, value in optimal_shipments.items():
        print(f"  {shipment.replace('Shipment_', '').replace('_', ' to ')}: {value} units")
    print(f"\nMinimum Total Delivery Cost = ${total_cost:,.2f}")
else:
    print("No optimal solution found.")


Status: Optimal

Optimal Shipments:
  Depot1 to Store1: 2000.0 units
  Depot1 to Store2: 500.0 units
  Depot2 to Store2: 1100.0 units
  Depot2 to Store3: 2000.0 units
  Depot3 to Store2: 1250.0 units

Minimum Total Delivery Cost = $812,500.00


##4. Reporting the return

In [ ]:
import pandas as pd

if status == 'Optimal':
    # Prepare data for DataFrame
    shipment_data = []
    for shipment_key, value in optimal_shipments.items():
        parts = shipment_key.split('_')
        depot = parts[1]
        store = parts[2]
        shipment_data.append({'Depot': depot, 'Store': store, 'Units': value})

    # Create DataFrame
    df_optimal_shipments = pd.DataFrame(shipment_data)

    # Pivot the DataFrame to get a 3x3 table format
    df_pivot_shipments = df_optimal_shipments.pivot_table(
        index='Depot', columns='Store', values='Units', fill_value=0
    ).astype(int) # Convert to integer for zero decimal places

    print("\nOptimal Shipments (Depot to Store):")
    display(df_pivot_shipments)

    print(f"\nMinimum cost = ${total_cost:,.0f}") # Format total_cost to zero decimal places and shorten text
else:
    print("No optimal solution found.")


Optimal Shipments (Depot to Store):


Store,Store1,Store2,Store3
Depot,,,
Depot1,2000,500,0
Depot2,0,1100,2000
Depot3,0,1250,0



Minimum cost = $812,500
